In [ ]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
from sage.libs.libecm import ecmfactor
from sage.misc.search import search
import time

def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f.
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*t]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*t])
    else:
        raise "l est inerte"

#La racine utilisée ici n'est pas le coefficient b de la forme quadratique associée.




def forme_de_norme(l,D):

    #Calcule une forme quadratique binaire primitive normale de coefficient dominant l, de discriminant D. 
    #On demande à ce que l soit décomposé dans l'ordre de discriminant D.
    
    if kronecker(D,l) == 1:
        if l == 2 :
            d = Mod(D,8)
            b = square_root_mod_prime_power(d,2,3)
            b = ZZ(b)
        else :
            x = Mod(D,4)
            d = Mod(D,l)
            y = square_root_mod_prime(d,l)
            b = x.crt(y)
            b = ZZ(b)
        ql = BinaryQF([l,b, int(int((b^2 - D))/int(4*l))])
        return ql
    else:
        raise ValueError('l pas split')

def formes_generatrices (D,N,test_norm): #Correspond à l'algorithme 5 du rapport.

# Trouver une famille génératrice du groupes de classes, vu comme formes quadratiques réduites.
# En n'utilisant que des idéaux de normes premiers autorisés pour le calcul d'isogénies horizontales.
# On génère simplement une famille de N idéaux, N supposé suffisament grand pour que leurs classes soient génératrices.
# N = 2log^2(D) suffit (Admet hypothèse de Riemann Généralisée).
    
    Idéaux = []
    Formes = []
    Primes = []
    i = 2
    while i < N:
        if kronecker(D,i) == 1 and (test_norm)%i != 0:
            Primes.append(i)
            qi = forme_de_norme(i,D)
            Li = NumberFieldOrderIdeal(O,qi)
            Idéaux.append(Li)
            Formes.append(qi)
        i = next_prime(i)
    return Idéaux, Formes, Primes



def factor_ecm(a,Primes):  

    #On utilise ECM pour vérifier que l'entier a se factorise uniquement avec les nombres premiers de la liste Primes. 
    #Si a est friable, on calcule sa décomposition en facteurs premiers.
    
    produit = a
    facteur = 1
    reste = a
    
    factorisation = []
    friable = True         # Dès que l'on trouve un facteur qui contredit la friabilité, on s'arrête. 
    count = 0              # On évite les boucles infinies. 

    while produit > 1 and friable:
        
        assert count < 1000
        
        # On teste si produit est un nombre premier.

        if produit.is_prime() :
            facteur = produit
            reste = 1 
         

        else:
            testecm = ecmfactor(produit, 0.00)
            #testecm[0] == False ssi pas de facteur trouvé. testecm[1] == produit ssi le facteur trouvé est trivial maximal.
            while (testecm[0] == False or testecm[1] == produit):
                if count < 1000:
                    testecm = ecmfactor(produit, 0.00)
                    count = count + 1
                else:
                    print('WARNING ECM ne trouve pas de facteur pour :', produit)  #Peut arriver sur de petits nombres, tester ecmfactor(9,0.00)
                    facto_brut = produit.factor()
                    testecm = (True, facto_brut[0][0], 0)
                    count = 0
            facteur = testecm[1]
            reste = produit//facteur

        # On vérifie que le facteur trouvé est premier, sinon on le factorise encore
        
        while facteur.is_prime() == False:
            if count < 1000:
                testecm2 = ecmfactor(facteur, 0.00)
                count = count + 1
            else:
                print('WARNING ECM facto_brut pour factoriser :', facteur)
                facto_brut = produit.factor()
                testecm2 = (True, facto_brut[0][0], 0)
                count = 0
            if testecm2[0] == True :
                facteur2 = testecm2[1]
                reste = reste*(facteur//facteur2)
                facteur = facteur2
    

        # On teste le facteur premier trouvé. Si il est dans Primes, on calcule sa valuation. 
        
        if search(Primes,facteur)[0]: #search est un algorithme de recherche dans une liste triée.
            exposant = 1
            while reste%facteur == 0:
                exposant = exposant + 1
                reste = reste//facteur
            factorisation.append([facteur, exposant])
        else:
            factorisation.append([facteur, 1])
            factorisation.append([reste, 1])
            friable = False
            
        produit = reste 
        
    return friable, factorisation




# Calcul dans le groupe de classes
# Version 1: Algorithme énoncé par Jao Soukarev. 

def factorisation(L,D,N,Borne_z,test_norm):     

    #On suppose L être un idéal de norme l premier décomposé.
    #On cherche à ramener la classe de L à une classe simplifiable en multipliant au hasard par des petits idéaux.
    #On représente les classes par des formes quadratiques réduites, et on cherche un formes de coefficients dominant friable.
    
    #Implémentation de l'algorithme énoncé par Jao et Soukharev.
    #Borne_z est une borne donnée par Jao dans l'analyse de l'algorithme. Le nombres de produits aléatoires ne dépassera pas Borne_z. 
    
    Idéaux, Formes, Primes = formes_generatrices(D,N,test_norm)
    k = len(Primes)
    
    ql = (L.quadratic_form()).reduced_form() #Représente la classe de L
    a = ql[0]
    relation = ql        # Partant de ql, on va chercher une classe simplifiable
    
    expx = [0]*k  # La liste des exposants donnants des produits aléatoires effectués
    liste_indice = []
    compte = 0
    test_facto, facto = factor_ecm(a,Primes)
    
    while test_facto == False:
        compte = compte + 1  
        assert compte < 1000  #sécurité arbitraire
        
        # Choix d'exposants au hasard, on veut entre 3 et Borne_z coefficients non nuls (Cf algo 3 Jao Soukharev)
        
        expx = [0]*k  # La liste des exposants donnants des produits aléatoires effectués
        nb_indice = randint(3,Borne_z) 
        
        liste_indice = []   # On choisit les idéaux qui vont intervenir dans la factorisation, au hasard.
        while len(liste_indice) < min(nb_indice,k):
            i = randint(0,k-1)
            if i not in liste_indice :
                liste_indice.append(i)
        
        for i in liste_indice:
            expi = randint(1,int((N/Primes[i])^2))   #On attribue à chaque indice un exposant xi, bornée d'après Jao Soukharev.
            expx[i] = expi

        # Calcul de ql*( fi^exp(xi) pour tout i ) 
        
        relation = ql
        for i in [0 .. k-1]:
            for _ in [1 .. (expx[i])]:
                relation = (relation*Formes[i]).reduced_form()
                
        a = relation[0] 
        test_facto, facto = factor_ecm(a,Primes)
         
    
    #Simplification de la classe "relation" trouvée

    expu = [0]*k     #liste des exposants de l'écriture de "relation" dans la famille génératrice
    ka = len(facto)
    b = relation[1]
    for j in [0 .. ka-1]:
        i = 0
        while Primes[i] != facto[j][0]:
            i = i+1
        expu[i] = facto[j][1]
        bi = Formes[i][1]
        if Mod( b , 2*Primes[i]) != Mod( bi , 2*Primes[i]):
                    expu[i] = -expu[i]
        
    
    #Inverser la relation pour trouver L:
    
    expe = []     #liste des exposants de l'écritures de L dans la famille génératrice, positifs.
    factoL = []    #Idéaux apparaissant dans l'écriture de L dans la famille génératrice.
    for i in [0 .. k-1] :
        ei = expu[i] - expx[i]
        if ei > 0 :
            expe.append(ei)
            factoL.append(Idéaux[i])
        elif ei < 0 :
            expe.append(-ei)
            factoL.append(Idéaux[i].conjugate())
            
    
    return expe, factoL, compte



# Version 2: Algorithme 6 du rapport.

def factorisation_2(L,D,N,Borne_t,test_norm):
    
    ##On suppose L être un idéal de norme l premier décomposé.
    #On cherche à ramener la classe de L à une classe simplifiable en multipliant au hasard par des petits idéaux.
    #On représente les classes par des formes quadratiques réduites, et on cherche un formes de coefficients dominant friable.
    
    #Borne_t est le nombre de pas nécessaire pour obtenir des marches aléatoires uniformes dans le groupe de classes d'idéaux.
    
    Idéaux, Formes, Primes = formes_generatrices (D,N,test_norm)
    k = len(Primes)
    
    ql = (L.quadratic_form()).reduced_form() #Représente la classe de L.
    a = ql[0]
    relation = ql        # Partant de ql, on va chercher une classe simplifiable.
    
    expx = [0]*k         # La liste des exposants donnants des produits aléatoires effectués.
    compte = 0
    test_facto, facto = factor_ecm(a,Primes)
    
    while test_facto == False:
        compte = compte + 1
        assert compte < 1000  #sécurité arbitraire
        
        #On effectue une marche aléatoire en partant de ql, de Borne_t pas. 
        
        expx = [0]*k   # La liste des exposants donnants des produits aléatoires effectués.
        for i in [1 .. Borne_t]:
            indice = randint(0,k-1)
            increment = randint(0,1)
            if increment == 0:
                increment = -1
            expx[indice] = expx[indice] + increment
            
            
        # Calcul de ql*( fi^exp(xi) pour tout i ) 
        
        relation = ql
        for i in [0 .. k-1]:
            facteur = Formes[i]
            exp = expx[i]
            if exp < 0:
                facteur = BinaryQF([facteur[0],-facteur[1],facteur[2]])
                exp = -exp
            for _ in [1 .. exp]:
                relation = (relation*facteur).reduced_form()   #Toujours réduire : bonne idée ??
        a = relation[0] #Si a factorise complétement dans Primes, c'est gagné avec Seysen
        test_facto, facto = factor_ecm(a,Primes)
    
    #Simplification de la classe "relation" trouvée:

    expu = [0]*k          #liste des exposants de l'écriture de "relation" dans la famille génératrice
    ka = len(facto)
    b = relation[1]
    for j in [0 .. ka-1]:
        i = 0
        while Primes[i] != facto[j][0]:
            i = i+1
        expu[i] = facto[j][1]
        bi = Formes[i][1]
        if Mod( b , 2*Primes[i]) != Mod( bi , 2*Primes[i]):
                    expu[i] = -expu[i]
        
    
    #Inverser la relation pour trouver L:
    
    expe = []     #liste des exposants de l'écritures de L dans la famille génératrice, positifs.
    factoL = []    #Idéaux apparaissant dans l'écriture de L dans la famille génératrice.
    for i in [0 .. k-1] :
        ei = expu[i] - expx[i]
        if ei > 0 :
            expe.append(ei)
            factoL.append(Idéaux[i])
        elif ei < 0 :
            expe.append(-ei)
            factoL.append(Idéaux[i].conjugate())
            
    
    return expe, factoL ,compte

In [ ]:
def test(K,O,l,N,Borne_z,Borne_t):
    #On enchaîne les algorithmes factorisation et factorisation2, en mesurant le temps de calcul
    
    dk = K.discriminant()
    f = O.conductor()
    D = f^2*dk
    
    L = ideal_de_norme(l,f,D)
    # On choisit test_norm de sorte toujours commencer une marche aléatoire
    ql = L.quadratic_form()
    #print(ql)
    ql = ql.reduced_form()
    #print(ql)
    test_norm = ql[0]
    #print('test_norm :', test_norm)
    
    début = time.time()
    Exposant, Idéaux, Compte = factorisation(L,D,N,Borne_z,test_norm)
    t1 = time.time() - début

    début = time.time()
    Exposant2, Idéaux2, Compte2 = factorisation_2(L,D,N,Borne_t,test_norm)
    t2 = time.time() - début

    return Exposant, t1, Compte, Idéaux, Exposant2, t2, Compte2, Idéaux2

In [ ]:
D = -3635657473865223      #Deux valeurs choisis au hasard. 
#D = −48221266355996583 

K.<t> = QuadraticField(D)
O = K.maximal_order()
D = O.discriminant()

print('Discriminant ordre :', D)

N = 2*int(log(-D,2))^2                 
z = 1/(2*sqrt(3))  
Borne_z = int(sqrt(log(-D/3,2))/z)     
Borne_t = int(log(-D)/log(log(-D)))

somme_exp = []
temps = []
tentatives = []
nb_idéaux = []
norme_max = []
nb_exp = []

somme_exp2 = []
temps2 = []
tentatives2 = []
nb_idéaux2 = []
norme_max2 = []
nb_exp2 = []

for i in range(30):
    l = next_prime(randint(10^20, 10^21))
    while kronecker(D,l) != 1:
        l = next_prime(l)
    exp1, t1, c1, Id1, exp2, t2, c2, Id2 = test(K,O,l,N,Borne_z,Borne_t)
    temps.append(t1)
    temps2.append(t2)
    tentatives.append(c1)
    tentatives2.append(c2)
    somme_exp.append(sum(exp1))
    somme_exp2.append(sum(exp2))
    nb_idéaux.append(len(Id1))
    nb_idéaux2.append(len(Id2))
    norme_max.append((Id1[-1]).norm())
    norme_max2.append((Id2[-1]).norm())

In [ ]:
import numpy as np
np.mean(somme_exp), np.mean(somme_exp2)

In [ ]:
np.mean(temps), np.mean(temps2)

In [ ]:
np.mean(tentatives), np.mean(tentatives2)

In [ ]:
np.mean(nb_idéaux), np.mean(nb_idéaux2)

In [ ]:
np.mean(norme_max), np.mean(norme_max2)